# 固定决策窗口的资金流接收证据

一次性实收采集。09:45/10:30两次读取，迟到与旧日返回不合格。采集中不声称结果完成，完成后同步中文结论与NAS冷读回。

In [ ]:
from pathlib import Path
from datetime import datetime,timezone
import os,json,hashlib,time
import numpy as np
import pandas as pd
import qlib
from qlib.workflow import R
from futu import OpenQuoteContext,SysConfig
import exchange_calendars as xc
from src.data.flow_receipts import capture_flow
from src.data.minute_intake import atomic_json
from src.data.qlib_native import build_qlib_store_from_frame,load_qlib_feature_frame,QlibProviderConfig
from src.data.nas_market_data import archive_nas_recorder
root=Path.cwd()
while not (root/'SPEC.md').exists():
 if root==root.parent:raise RuntimeError('repository directory required')
 root=root.parent
os.chdir(root);os.environ['MLFLOW_ALLOW_FILE_STORE']='true';card=root/'research/flow_receipts_20261014';plan=json.loads((card/'plan.json').read_text());plan_hash=hashlib.sha256((card/'plan.json').read_bytes()).hexdigest();assert xc.get_calendar('XNYS').is_session(plan['as_of']);qlib.init(provider_uri=str(root/plan['bootstrap_provider']),region='us');tracking=card/'qlib_recorder';out=root/plan['raw_directory'];receipts=[];raw={}
summary={'experiment':card.name,'decision':'collecting_fixed_live_receipts','status':'running','objective_achieved':False,'new_fits':0,'new_backtests':0,'new_orders':0,'forecast_input_qualified':False,'provider_uri':str(root/plan['bootstrap_provider']),'region':'us','as_of':plan['as_of'],'artifact_policy':{'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/*','sources/*']}}
with R.start(experiment_name=card.name,recorder_name='fixed_live_flow_receipts',uri=str(tracking)):
 rec=R.get_recorder();summary.update(recorder_id=rec.id,experiment_id=str(rec.experiment_id),recorder_uri=str(tracking));rec.save_objects(artifact_path='inputs',plan=plan);rec.save_objects(artifact_path='sources',receipt_contract=(root/'src/data/flow_receipts.py').read_text());rec.log_metrics(registered_artifact_count=2,captured_requests=0);atomic_json(summary,card/'summary.json');atomic_json({'kernel_pid':os.getpid(),'parent_pid':os.getppid(),'plan_sha256':plan_hash,'started_at_utc':datetime.now(timezone.utc).isoformat(),'recorder_id':rec.id},card/'runtime_process.json')
 SysConfig.set_all_thread_daemon(True);context=OpenQuoteContext(host='127.0.0.1',port=11111)
 try:
  for slot in plan['slots']:
   while pd.Timestamp.now(tz='UTC')<pd.Timestamp(slot['request_at_utc']):
    remaining=(pd.Timestamp(slot['request_at_utc'])-pd.Timestamp.now(tz='UTC')).total_seconds();atomic_json({'state':'waiting_for_fixed_slot','next_slot':slot['name'],'captured_requests':len(receipts),'observed_at_utc':datetime.now(timezone.utc).isoformat()},card/'progress.json');time.sleep(min(30.,max(.01,remaining)))
   current=[]
   for symbol in plan['symbols']:
    r=capture_flow(context,symbol=symbol,slot=slot,output_dir=out);receipts.append(r);current.append(r)
    if r.get('path'):raw[slot['name']+'_'+symbol]=pd.read_parquet(out/r['path'])
   rec.save_objects(artifact_path='inputs/'+slot['name'],receipts=current,raw={k:v for k,v in raw.items() if k.startswith(slot['name']+'_')});rec.log_metrics(captured_requests=len(receipts));atomic_json({'state':'slot_captured','slot':slot['name'],'captured_requests':len(receipts),'statuses':[r['status'] for r in current]},card/'progress.json');print(slot['name'],[r['status'] for r in current],flush=True)
 finally:context.close()
 fields=['in_flow','super_in_flow','big_in_flow','mid_in_flow','sml_in_flow'];frames=[]
 for key,d in raw.items():
  if d.empty:continue
  f=d[fields].apply(pd.to_numeric,errors='coerce');f['datetime']=pd.to_datetime(d.capital_flow_item_time);f['instrument']=key.upper();frames.append(f)
 if not frames:raise RuntimeError('needs_data: no source observations for native readback')
 frame=pd.concat(frames,ignore_index=True);provider=root/plan['provider'];manifest=build_qlib_store_from_frame(frame,provider_uri=provider,freq='1min',as_of=plan['as_of'],field_map={f:f for f in fields},source='Immutable as-received classified-flow snapshots, separate vintages; cumulative point timestamps, not closed OHLCV bars')
 qlib.init(provider_uri=str(provider),region='us');native=load_qlib_feature_frame(QlibProviderConfig(provider,freq='1min'),instruments=frame.instrument.unique().tolist(),fields=['$'+f for f in fields],start=str(frame.datetime.min()),end=str(frame.datetime.max()));expected=frame.set_index(['instrument','datetime'])
 for f in fields:np.testing.assert_allclose(native['$'+f].reindex(expected.index),expected[f],rtol=2e-7,atol=.01,equal_nan=True)
 revisions=[]
 for window in ['open','recovery']:
  for symbol in plan['symbols']:
   a=raw.get(window+'_a_'+symbol);b=raw.get(window+'_b_'+symbol)
   if a is None or b is None:continue
   a=a.set_index('capital_flow_item_time');b=b.set_index('capital_flow_item_time');common=a.index.intersection(b.index);diff=pd.to_numeric(b.loc[common,'in_flow'])-pd.to_numeric(a.loc[common,'in_flow']);revisions.append({'window':window,'symbol':symbol,'overlap_points':len(common),'changed_points_above_1cent':int(diff.abs().gt(.01).sum()),'max_absolute_change':float(diff.abs().max())})
 target=[r for r in receipts if r['slot']['name']!='baseline'];summary.update(status='completed',decision='one_session_receipt_qualification_only',provider_uri=str(provider),target_receipts=len(target),timestamp_qualified_target_receipts=sum(r['qualified'] for r in target),scope=plan['scope'],qlib_docs='https://qlib.readthedocs.io/en/latest/component/data.html')
 rec.save_objects(artifact_path='analysis',receipts=receipts,revisions=pd.DataFrame(revisions),native=native,provider_manifest=manifest,summary=summary);artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';rec.log_metrics(registered_artifact_count=sum(p.is_file() for p in artifacts.rglob('*')))
artifacts=tracking/str(rec.experiment_id)/rec.id/'artifacts';atomic_json({'recorder_id':rec.id,'entries':[{'path':str(p.relative_to(card)),'sha256':hashlib.sha256(p.read_bytes()).hexdigest()} for p in sorted(artifacts.rglob('*')) if p.is_file()]},card/'verification_manifest.json');summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));atomic_json(summary,card/'summary.json');atomic_json(receipts,card/'receipts.json');pd.DataFrame(revisions).to_csv(card/'snapshot_revisions.csv',index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':summary['provider_uri'],'recorder_uri':summary['recorder_uri'],'artifact_policy':summary['artifact_policy']},sort_keys=False));assert hashlib.sha256((root/'production/stage1_futu_sim_v1/runtime_release.json').read_bytes()).hexdigest()==plan['release_sha256'];atomic_json({'state':'completed','captured_requests':len(receipts)},card/'progress.json');print(summary)


# 固定窗口实收采集进行中，尚未完成实验

已冻结2026-10-14纽约09:45:10/09:45:40和10:30:10/10:30:40两个窗口，每次依次读取SOXX/SOXL。响应必须分别早于09:46、10:31；错过不补采，旧日/重复/未来源时间不合格，原回执不被重跑覆盖。

一次性notebook进程已经实际启动。盘前基线两份响应均为2026-10-13旧日，按规则不合格；尚未取得正式窗口结果。资金流last_valid_time比本机接收时间快约9秒；另行三次只读全局时钟核查显示服务器秒级时间与本机一致、OpenD本机时间落在请求接收区间内。该差异不能直接归为机器时钟偏移，也不能把该字段当成历史首次发布时间。固定规则暂不修改，原数据与接收时间分别保留。

本卡status=running；notebook尚在执行，未形成完整分析、最终NAS归档或冷读回验证。这里不是完成结果或晋升卡。只读收集，无模型训练、收益检验或订单，稳定净超额目标未完成。完成固定窗口后需同步执行结果、中文结论、summary与NAS证据，再交付最终结论。

运行时PID与进度文件仅用于定位；后续必须通过实际进程/工具session核实存活，不因等待超时重启或重复采集。截止北京时间22:31之后，再核对任务实际终态。
